# Aula 1, Pipeline de RAG

Notebook executável que acompanha a aula [01-pipeline-de-rag.md](../../lessons/modulo-09-rag/01-pipeline-de-rag.md).

## O que vamos fazer aqui

Vamos montar um RAG mínimo de ponta a ponta, sem bibliotecas pesadas: uma base de notas de aula,
vetores TF-IDF como embeddings, busca por cosseno para recuperar o trecho relevante e a montagem
do prompt que entregaria esse trecho ao LLM. Tudo em Python puro.

## Indexação e recuperação

Indexamos os documentos como vetores TF-IDF e recuperamos o trecho de maior similaridade do
cosseno com a pergunta.

In [ ]:
import re
import math
from collections import Counter

documentos = [
    "A derivada mede a taxa de variação instantânea de uma função em um ponto.",
    "A regra da cadeia permite derivar funções compostas multiplicando as derivadas.",
    "Uma matriz é uma tabela de números organizada em linhas e colunas.",
    "Em Python, uma função é definida com a palavra-chave def.",
]


def tokenizar(texto):
    return re.findall(r"\w+", texto.lower())


N = len(documentos)
df = Counter()
for d in documentos:
    for w in set(tokenizar(d)):
        df[w] += 1
idf = {w: math.log(N / f) for w, f in df.items()}


def vetor(texto):
    tf = Counter(tokenizar(texto))
    return {w: tf[w] * idf.get(w, 0.0) for w in tf}


def cosseno(a, b):
    prod = sum(a[w] * b.get(w, 0.0) for w in a)
    na = math.sqrt(sum(v * v for v in a.values()))
    nb = math.sqrt(sum(v * v for v in b.values()))
    return prod / (na * nb) if na and nb else 0.0


base = [vetor(d) for d in documentos]


def recuperar(pergunta, k=1):
    q = vetor(pergunta)
    ranking = sorted(((cosseno(q, base[i]), i) for i in range(N)), reverse=True)
    return [documentos[i] for _, i in ranking[:k]]


print("Trecho recuperado:", recuperar("O que é a derivada de uma função?", k=1)[0])

A busca recupera o trecho sobre a derivada, o mais relevante para a pergunta.

## Montando o prompt com contexto e gerando a resposta

Juntamos o trecho recuperado à pergunta em um prompt que instrui o modelo a responder apenas com
base no contexto. A geração via Ollama é opcional.

In [ ]:
def montar_prompt(pergunta, trechos):
    contexto = "\n".join(f"- {t}" for t in trechos)
    return (
        "Use apenas o contexto abaixo para responder.\n"
        f"Contexto:\n{contexto}\n\n"
        f"Pergunta: {pergunta}\nResposta:"
    )


pergunta = "O que é a derivada de uma função?"
prompt = montar_prompt(pergunta, recuperar(pergunta, k=1))
print(prompt)

try:
    import ollama
    r = ollama.chat(model="llama3.1", messages=[{"role": "user", "content": prompt}])
    print("\nResposta do LLM:\n", r["message"]["content"].strip())
except Exception as erro:
    print("\n(Ollama não disponível; o prompt acima é o que iria ao modelo.)")

Esse é o RAG inteiro em miniatura: indexar, recuperar e aumentar o prompt. As próximas
aulas melhoram cada peça. Para o projeto, acrescente documentos e teste perguntas que dependam
deles.